In [1]:
import psycopg2
import datetime as dt
import numpy as np
import pandas as pd
import logging
import json

In [2]:
# Configure logging
logging.basicConfig(filename='model_training.log', level=logging.DEBUG, 
                    format='%(asctime)s %(levelname)s:%(message)s')

In [3]:
# Load database configuration from JSON
with open('../../db_config.json') as config_file:
    db_params = json.load(config_file)
 
# Connect to the database
try:
    conn = psycopg2.connect(**db_params)
    cursor = conn.cursor()
    logging.info('Connection to database established')
except OperationalError as e:
    logging.error('Connection to database could not be established.')

INFO: Connection to database established


In [4]:
query = '''
-- Filtered Base Data View
CREATE OR REPLACE VIEW group14_warehouse.filtered_base_data AS
SELECT *
FROM data_lake.safe_driving
WHERE is_valid = 'True'
  AND TRIM(incident_severity) IN ('HC1', 'HC2', 'SP1', 'SP2', 'HB1', 'HA1');
'''
cursor.execute(query)
conn.commit()

In [5]:
query = '''
-- Accident Data Renamed View
CREATE OR REPLACE VIEW group14_warehouse.accident_data_renamed AS
SELECT "Year" AS year,
       "Accident severity" AS accident_severity,
       municipality,
       town,
       "First Mode of Transport" AS first_mode_of_transport,
       "Second mode of Transport" AS second_mode_of_transport,
       "Area Type" AS area_type,
       "Light condition" AS light_condition,
       "Road condition" AS road_condition,
       "Road surface" AS road_surface,
       "Road situation" AS road_situation,
       "Speed limit" AS speed_limit,
       street,
       weather,
       accidents
FROM data_lake.accident_data_17_23;
'''

cursor.execute(query)
conn.commit()

In [6]:
query = '''
-- Accident Probability View
CREATE OR REPLACE VIEW group14_warehouse.accident_probability AS
SELECT a.incident_year,
       a.incident_severity,
       a.incident_cnt,
       b.accident_severity,
       b.accident_cnt,
       100 * (b.accident_cnt::float / a.incident_cnt::float) AS accident_probability
FROM (SELECT EXTRACT(YEAR FROM event_start) AS incident_year,
             TRIM(incident_severity)          AS incident_severity,
             COUNT(*)                         AS incident_cnt
      FROM group14_warehouse.filtered_base_data
      GROUP BY 1, 2) a
         JOIN (SELECT year AS accident_year,
                      accident_severity,
                      SUM(accidents) AS accident_cnt
               FROM group14_warehouse.accident_data_renamed
               WHERE year >= 2018
               GROUP BY 1, 2
               ORDER BY 1, 2) b
              ON a.incident_year = b.accident_year;
'''

cursor.execute(query)
conn.commit()

In [7]:
query = '''
-- Final Accident Incident Merged Table
DROP TABLE IF EXISTS group14_warehouse.accident_incident_merged_table;
CREATE TABLE group14_warehouse.accident_incident_merged_table AS
SELECT eventid,
       event_start,
       event_end,
       duration_seconds,
       latitude,
       longitude,
       speed_kmh,
       end_speed_kmh,
       maxwaarde,
       category,
       a.incident_severity,
       b.accident_severity,
       is_valid,
       road_segment_id,
       road_manager_type,
       road_number,
       road_name,
       place_name,
       municipality_name,
       road_manager_name,
       b.accident_probability,
       c.light_condition,
       c.speed_limit,
       c.street
FROM group14_warehouse.filtered_base_data a
         LEFT JOIN group14_warehouse.accident_probability b
                   ON EXTRACT(YEAR FROM a.event_start) = b.incident_year
                       AND TRIM(a.incident_severity) = b.incident_severity
         LEFT JOIN group14_warehouse.accident_data_renamed c
                   ON LOWER(a.road_name) = LOWER(c.street)
                       AND b.accident_severity = c.accident_severity
                       AND EXTRACT(YEAR FROM a.event_start) = c.year;
'''

cursor.execute(query)
conn.commit()

In [8]:
query = '''
SELECT * FROM group14_warehouse.accident_incident_merged_table;
'''

cursor.execute(query)
df = pd.DataFrame(cursor.fetchall(), columns=[ "eventid", "event_start", "event_end", "duration_seconds",
                                               "latitude", "longitude", "speed_kmh", "end_speed_kmh", "maxwaarde",
                                               "category", "incident_severity", "accident_severity", "is_valid",
                                               "road_segment_id", "road_manager_type", "road_number", "road_name",
                                               "place_name", "municipality_name", "road_manager_name", "accident_probability",
                                               "light_condition", "speed_limit", "street"])
print(df.head())

     eventid             event_start               event_end  \
0  109896417 2023-02-14 17:09:19.100 2023-02-14 17:09:20.300   
1  109909237 2023-02-14 17:26:31.000 2023-02-14 17:26:46.000   
2  109932114 2023-02-14 18:02:23.000 2023-02-14 18:02:39.000   
3  109863283 2023-02-14 07:34:55.700 2023-02-14 07:34:56.800   
4  109135058 2023-02-14 18:50:58.400 2023-02-14 18:50:58.800   

   duration_seconds  latitude  longitude  speed_kmh  end_speed_kmh  maxwaarde  \
0               1.2  51.57601   4.792040  21.747630      26.330444   0.746736   
1              15.0  51.57192   4.762366  64.373760      64.373760  70.811134   
2              16.0  51.57219   4.765710  62.128902      67.266106  70.225320   
3               1.1  51.58215   4.820879  35.405567      37.014910   0.828812   
4               0.4  51.59386   4.755197  33.796223       3.218688   0.854562   

          category  ... road_manager_type road_number               road_name  \
0  HARSH CORNERING  ...                 G      

In [11]:
df["event_start"] = pd.to_datetime(df["event_start"])

print(df.sort_values("event_start")[["eventid", "event_start", "event_end", "duration_seconds",
                         "latitude", "longitude", "speed_kmh", "end_speed_kmh", "maxwaarde"]].head())

         eventid             event_start               event_end  \
1765386  5653658 2018-01-01 00:18:20.500 2018-01-01 00:18:28.500   
1765406  5653658 2018-01-01 00:18:20.500 2018-01-01 00:18:28.500   
1765407  5653658 2018-01-01 00:18:20.500 2018-01-01 00:18:28.500   
1765408  5653658 2018-01-01 00:18:20.500 2018-01-01 00:18:28.500   
1765409  5653658 2018-01-01 00:18:20.500 2018-01-01 00:18:28.500   

         duration_seconds  latitude  longitude  speed_kmh  end_speed_kmh  \
1765386               8.0  51.59962   4.749473  82.076546      82.076546   
1765406               8.0  51.59962   4.749473  82.076546      82.076546   
1765407               8.0  51.59962   4.749473  82.076546      82.076546   
1765408               8.0  51.59962   4.749473  82.076546      82.076546   
1765409               8.0  51.59962   4.749473  82.076546      82.076546   

         maxwaarde  
1765386   86.90458  
1765406   86.90458  
1765407   86.90458  
1765408   86.90458  
1765409   86.90458  


In [12]:
print(df.sort_values("event_start")[["eventid", "event_start", "category", "incident_severity", "accident_severity", "is_valid",
                         "road_segment_id", "road_manager_type", "road_number", "road_name"]].head())

         eventid             event_start          category incident_severity  \
1765386  5653658 2018-01-01 00:18:20.500  SPEED                        SP1     
1765406  5653658 2018-01-01 00:18:20.500  SPEED                        SP1     
1765407  5653658 2018-01-01 00:18:20.500  SPEED                        SP1     
1765408  5653658 2018-01-01 00:18:20.500  SPEED                        SP1     
1765409  5653658 2018-01-01 00:18:20.500  SPEED                        SP1     

            accident_severity  is_valid  road_segment_id road_manager_type  \
1765386                 Fatal      True        600750243                 G   
1765406  Material Damage Only      True        600750243                 G   
1765407  Material Damage Only      True        600750243                 G   
1765408  Material Damage Only      True        600750243                 G   
1765409  Material Damage Only      True        600750243                 G   

        road_number          road_name  
1765386  

In [13]:
print(df.sort_values("event_start")[["eventid", "event_start", "place_name", "municipality_name", "road_manager_name", "accident_probability",
                         "light_condition", "speed_limit", "street"]].head())

         eventid             event_start place_name municipality_name  \
1765386  5653658 2018-01-01 00:18:20.500      Breda             Breda   
1765406  5653658 2018-01-01 00:18:20.500      Breda             Breda   
1765407  5653658 2018-01-01 00:18:20.500      Breda             Breda   
1765408  5653658 2018-01-01 00:18:20.500      Breda             Breda   
1765409  5653658 2018-01-01 00:18:20.500      Breda             Breda   

        road_manager_name  accident_probability light_condition speed_limit  \
1765386             Breda              0.019675        Daylight     70 km/h   
1765406             Breda              3.049560        Daylight     70 km/h   
1765407             Breda              3.049560        Daylight     70 km/h   
1765408             Breda              3.049560        Daylight     70 km/h   
1765409             Breda              3.049560        Daylight     70 km/h   

                    street  
1765386  Backer en Ruebweg  
1765406  Backer en Ruebweg  